In [1]:
import pandas as pd 

train_df = pd.read_csv("datasets/titanic/train.csv")
test_df = pd.read_csv("datasets/titanic/test.csv")

In [2]:
print(train_df.shape)
train_df.head(10)

(891, 12)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,0,3,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,1,3,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,1,2,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


In [3]:
print(test_df.shape)
test_df.head(10)

(418, 11)


,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
5,897,3,"Svensson, Mr. Johan Cervin",male,14.0,0,0,7538,9.2250,NaN,S
6,898,3,"Connolly, Miss. Kate",female,30.0,0,0,330972,7.6292,NaN,Q
7,899,2,"Caldwell, Mr. Albert Francis",male,26.0,1,1,248738,29.0000,NaN,S
8,900,3,"Abrahim, Mrs. Joseph (Sophie Halaut Easu)",female,18.0,0,0,2657,7.2292,NaN,C
9,901,3,"Davies, Mr. John Samuel",male,21.0,2,0,A/4 48871,24.1500,NaN,S


In [4]:
train_df["Cabin"].value_counts()

Cabin
G6             4
C23 C25 C27    4
B96 B98        4
F33            3
E101           3
              ..
E17            1
A24            1
C50            1
B42            1
C148           1
Name: count, Length: 147, dtype: int64

In [5]:
from sklearn.preprocessing import LabelEncoder

train_df_prep = train_df.copy()
le= LabelEncoder()
train_df_prep["Sex"] = le.fit_transform(train_df_prep["Sex"])

le_2 = LabelEncoder()
train_df_prep["Embarked"] = le_2.fit_transform(train_df_prep["Embarked"])

train_df_prep = train_df_prep.drop(["PassengerId"],axis=1)
train_df_prep = train_df_prep.drop(["Name"],axis=1)

In [6]:
train_df_prep.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,1,22.0,1,0,A/5 21171,7.2500,NaN,2
1,1,1,0,38.0,1,0,PC 17599,71.2833,C85,0
2,1,3,0,26.0,0,0,STON/O2. 3101282,7.9250,NaN,2
3,1,1,0,35.0,1,0,113803,53.1000,C123,2
4,0,3,1,35.0,0,0,373450,8.0500,NaN,2


In [7]:
le3 = LabelEncoder()
combined_tickets = pd.concat([train_df_prep["Ticket"], test_df["Ticket"]])
le3.fit(combined_tickets)
train_df_prep["Ticket"] = le3.transform(train_df_prep["Ticket"])

le4 = LabelEncoder()
combined_cabin = pd.concat([train_df_prep["Cabin"], test_df["Cabin"]])
le4.fit(combined_cabin)
train_df_prep["Cabin"] = le4.transform(train_df_prep["Cabin"])

In [8]:
features = train_df_prep.columns.tolist()
features.remove("Cabin")
features

['Survived',
 'Pclass',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Embarked']

In [9]:
cabin_train_df = train_df_prep[train_df_prep["Cabin"] != 186]
cabin_test_df = train_df_prep[train_df_prep["Cabin"] == 186]

In [10]:
cabin_train_df.shape

(204, 10)

In [11]:
cabin_test_df.shape

(687, 10)

In [12]:
X_train = cabin_train_df[features]
y_train = cabin_train_df["Cabin"].values

X_test = cabin_test_df[features]
y_test = cabin_test_df["Cabin"].values

In [13]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="mean")
X_train[["Age","Fare"]] = imputer.fit_transform(X_train[["Age","Fare"]])
X_test[["Age","Fare"]] = imputer.transform(X_test[["Age","Fare"]])

In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train[["Age","Fare"]] = scaler.fit_transform(X_train[["Age","Fare"]])
X_test[["Age","Fare"]] = scaler.transform(X_test[["Age","Fare"]])

In [15]:
from sklearn.ensemble import RandomForestClassifier

rfc = RandomForestClassifier()
rfc.fit(X_train, y_train)

C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\sklearn\ensemble\_forest.py:830: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)
C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\sklearn\tree\_classes.py:297: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)
C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\sklearn\tree\_classes.py:297: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)
C:\Users\Berkay\Desktop\ML_projects\LSTM_sentiment\Lib\site-packages\sklearn\tree\_classes.py:297: UserWarning: The nu

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [16]:
y_hat = rfc.predict(X_test)

In [17]:
cabin_test_df.loc[cabin_test_df["Cabin"] == 186, "Cabin"] = y_hat

In [18]:
prep_df = pd.concat([cabin_train_df, cabin_test_df]).sort_index()

In [19]:
prep_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,0,3,1,22.0,1,0,720,7.2500,179,2
1,1,1,0,38.0,1,0,816,71.2833,106,0
2,1,3,0,26.0,0,0,914,7.9250,147,2
3,1,1,0,35.0,1,0,65,53.1000,70,2
4,0,3,1,35.0,0,0,649,8.0500,182,2
...,...,...,...,...,...,...,...,...,...,...
886,0,2,1,27.0,0,0,130,13.0000,143,2
887,1,1,0,19.0,0,0,18,30.0000,40,2
888,0,3,0,NaN,1,2,923,23.4500,184,2
889,1,1,1,26.0,0,0,11,30.0000,77,0


In [20]:
prep_df.isna().sum()

Survived      0
Pclass        0
Sex           0
Age         177
SibSp         0
Parch         0
Ticket        0
Fare          0
Cabin         0
Embarked      0
dtype: int64

In [21]:
imputer = SimpleImputer(strategy="mean")
prep_df["Age"] = imputer.fit_transform(prep_df[["Age"]])

scaler = StandardScaler()
prep_df[["Age","Fare"]] = scaler.fit_transform(prep_df[["Age","Fare"]])

In [22]:
features = prep_df.columns.tolist()
features.remove("Survived")
features

['Pclass',
 'Sex',
 'Age',
 'SibSp',
 'Parch',
 'Ticket',
 'Fare',
 'Cabin',
 'Embarked']

In [23]:
X = prep_df[features]
y = prep_df["Survived"]

In [24]:
test_df

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...
413,1305,3,"Spector, Mr. Woolf",male,NaN,0,0,A.5. 3236,8.0500,NaN,S
414,1306,1,"Oliva y Ocana, Dona. Fermina",female,39.0,0,0,PC 17758,108.9000,C105,C
415,1307,3,"Saether, Mr. Simon Sivertsen",male,38.5,0,0,SOTON/O.Q. 3101262,7.2500,NaN,S
416,1308,3,"Ware, Mr. Frederick",male,NaN,0,0,359309,8.0500,NaN,S


In [25]:
test_df["Cabin"].value_counts()

Cabin
B57 B59 B63 B66    3
B45                2
C78                2
C31                2
C23 C25 C27        2
                  ..
B41                1
C7                 1
D40                1
D38                1
C105               1
Name: count, Length: 76, dtype: int64

In [26]:
test_df["Sex"] = le.transform(test_df["Sex"])
test_df["Embarked"] = le_2.transform(test_df["Embarked"])
test_df["Ticket"] = le3.transform(test_df["Ticket"])

test_df["Age"] = imputer.transform(test_df[["Age"]])
test_df[["Age","Fare"]] = scaler.transform(test_df[["Age","Fare"]])

test_df = test_df.drop(["PassengerId"],axis=1)
test_df = test_df.drop(["Name"],axis=1)

In [44]:
import numpy as np

test_df_cab = test_df[test_df["Cabin"].isna()]